<a href="https://colab.research.google.com/github/Piyush-Sharma-1/Wids_Endterm/blob/main/stock_sentiment_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stock Sentiment Prediction — AAPL, MSFT, GOOGL, AMZN, META

Trying to see if news sentiment can actually predict whether a stock goes up or down the next day. Using both VADER and FinBERT for sentiment scoring, pulling price data from yfinance and headlines from Google News RSS.

Pooled 5 stocks together over 5 years to get a decent sample size instead of relying on just one ticker.

## Setup

In [17]:

!pip install yfinance nltk beautifulsoup4 requests xgboost transformers torch --quiet

In [18]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

import yfinance as yf

import nltk
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

## Config — change tickers/years here if needed

In [19]:
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]
YEARS_BACK = 5
CHUNK_DAYS = 90  # query news in ~quarterly windows to page through history

from datetime import datetime, timedelta
END_DATE = datetime.today()
START_DATE = END_DATE - timedelta(days=365 * YEARS_BACK)

print("Date range:", START_DATE.date(), "to", END_DATE.date())
print("Tickers:", TICKERS)

def generate_date_chunks(start, end, chunk_days):
    chunks = []
    cur = start
    while cur < end:
        nxt = min(cur + timedelta(days=chunk_days), end)
        chunks.append((cur, nxt))
        cur = nxt
    return chunks

Date range: 2021-07-12 to 2026-07-11
Tickers: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']


## Getting price history for each stock

Also adding a few basic price features (lagged returns, rolling volatility) so I can check later whether sentiment actually adds anything beyond just price momentum.

In [20]:
def get_price_data(ticker, start, end):
    df = yf.download(ticker, start=start.strftime("%Y-%m-%d"), end=end.strftime("%Y-%m-%d"),
                      progress=False, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.columns.name = None
    df = df[["Close"]].copy()
    df["date"] = df.index.date
    df = df.reset_index(drop=True)

    df["return_1d"] = df["Close"].pct_change()
    df["return_lag_1"] = df["return_1d"].shift(1)
    df["return_lag_2"] = df["return_1d"].shift(2)
    df["volatility_5d"] = df["return_1d"].rolling(5).std()
    df["ticker"] = ticker
    return df

price_data = {t: get_price_data(t, START_DATE, END_DATE) for t in TICKERS}
price_data["AAPL"].head()

,Close,date,return_1d,return_lag_1,return_lag_2,volatility_5d,ticker
0,140.851639,2021-07-12,NaN,NaN,NaN,NaN,AAPL
1,141.962875,2021-07-13,0.007889,NaN,NaN,NaN,AAPL
2,145.384247,2021-07-14,0.024100,0.007889,NaN,NaN,AAPL
3,144.731155,2021-07-15,-0.004492,0.024100,0.007889,NaN,AAPL
4,142.693909,2021-07-16,-0.014076,-0.004492,0.024100,NaN,AAPL


## Pulling headlines from Google News RSS

Querying in date chunks since RSS only gives a small window of recent results per request — this loops through the full 5 years chunk by chunk.

In [21]:
def get_news_chunk(ticker, start_str, end_str, max_retries=3):
    query = f"{ticker}+stock+after:{start_str}+before:{end_str}"
    url = f"https://news.google.com/rss/search?q={query}"
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=10)
            soup = BeautifulSoup(resp.content, "xml")
            items = soup.find_all("item")
            rows = [{"title": i.title.text, "date": i.pubDate.text, "ticker": ticker} for i in items]
            return pd.DataFrame(rows)
        except Exception as e:
            print(f"Retry {attempt+1} for {ticker} [{start_str} to {end_str}]: {e}")
            time.sleep(2)
    return pd.DataFrame(columns=["title", "date", "ticker"])

def get_news_full_history(ticker, start, end, chunk_days=CHUNK_DAYS):
    chunks = generate_date_chunks(start, end, chunk_days)
    all_rows = []
    for c_start, c_end in chunks:
        df = get_news_chunk(ticker, c_start.strftime("%Y-%m-%d"), c_end.strftime("%Y-%m-%d"))
        all_rows.append(df)
        time.sleep(1.5)  # be polite to the endpoint, avoid rate-limiting
    combined = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame(columns=["title","date","ticker"])
    return combined

news_frames = []
for t in TICKERS:
    df = get_news_full_history(t, START_DATE, END_DATE)
    print(f"{t}: {len(df)} headlines fetched across {len(generate_date_chunks(START_DATE, END_DATE, CHUNK_DAYS))} date chunks")
    news_frames.append(df)

news_df = pd.concat(news_frames, ignore_index=True)
news_df = news_df.drop_duplicates(subset=["ticker", "title", "date"])
news_df["date"] = pd.to_datetime(news_df["date"]).dt.date
print("Total headlines:", len(news_df))
news_df.head()

AAPL: 1056 headlines fetched across 21 date chunks
MSFT: 1055 headlines fetched across 21 date chunks
GOOGL: 1418 headlines fetched across 21 date chunks
AMZN: 1343 headlines fetched across 21 date chunks
META: 1567 headlines fetched across 21 date chunks
Total headlines: 6405


,title,date,ticker
0,Apple (AAPL) Option Traders Bearish After Earn...,2021-08-02,AAPL
1,Warren Buffett Is Selling These 10 Stocks - Ya...,2021-08-23,AAPL
2,Binance ditches 'stock tokens' as global crack...,2021-07-16,AAPL
3,Opinion: Apple’s blowout earnings didn’t help ...,2021-07-28,AAPL
4,Apple (AAPL) stock down over 3% on App Store i...,2021-09-10,AAPL


##  VADER Sentiment

In [22]:
sia = SentimentIntensityAnalyzer()
news_df["vader_score"] = news_df["title"].apply(lambda x: sia.polarity_scores(x)["compound"])
news_df[["ticker", "title", "vader_score"]].head()

,ticker,title,vader_score
0,AAPL,Apple (AAPL) Option Traders Bearish After Earn...,0.0000
1,AAPL,Warren Buffett Is Selling These 10 Stocks - Ya...,0.0000
2,AAPL,Binance ditches 'stock tokens' as global crack...,0.0000
3,AAPL,Opinion: Apple’s blowout earnings didn’t help ...,0.4019
4,AAPL,Apple (AAPL) stock down over 3% on App Store i...,0.0000


## FinBERT sentiment

Using ProsusAI/finbert. Converting the pos/neg/neutral probabilities into a single signed score (pos - neg) so it's comparable to VADER's compound score.

In [26]:
FINBERT_MODEL = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
model.eval()

# FinBERT label order: 0=positive, 1=negative, 2=neutral (check model.config.id2label to confirm)
print(model.config.id2label)

def finbert_score_batch(texts, batch_size=16):
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=64)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = F.softmax(logits, dim=1).numpy()
        # signed score = P(positive) - P(negative), using id2label to find the right columns
        id2label = model.config.id2label
        pos_idx = [k for k, v in id2label.items() if v.lower() == "positive"][0]
        neg_idx = [k for k, v in id2label.items() if v.lower() == "negative"][0]
        batch_scores = probs[:, pos_idx] - probs[:, neg_idx]
        scores.extend(batch_scores.tolist())
    return scores

news_df["finbert_score"] = finbert_score_batch(news_df["title"].tolist())
news_df[["ticker", "title", "vader_score", "finbert_score"]].head()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{0: 'positive', 1: 'negative', 2: 'neutral'}


,ticker,title,vader_score,finbert_score
0,AAPL,Apple (AAPL) Option Traders Bearish After Earn...,0.0000,-0.813288
1,AAPL,Warren Buffett Is Selling These 10 Stocks - Ya...,0.0000,-0.002165
2,AAPL,Binance ditches 'stock tokens' as global crack...,0.0000,-0.253060
3,AAPL,Opinion: Apple’s blowout earnings didn’t help ...,0.4019,-0.759196
4,AAPL,Apple (AAPL) stock down over 3% on App Store i...,0.0000,-0.962453


##  Daily Sentiment Aggregation (per ticker)

In [27]:
daily_sentiment = (
    news_df.groupby(["ticker", "date"])
    .agg(
        vader_mean=("vader_score", "mean"),
        vader_std=("vader_score", "std"),
        finbert_mean=("finbert_score", "mean"),
        finbert_std=("finbert_score", "std"),
        news_count=("title", "count"),
    )
    .reset_index()
)
daily_sentiment[["vader_std", "finbert_std"]] = daily_sentiment[["vader_std", "finbert_std"]].fillna(0)
daily_sentiment.head()

,ticker,date,vader_mean,vader_std,finbert_mean,finbert_std,news_count
0,AAPL,2021-07-12,0.6369,0.0,0.038474,0.0,1
1,AAPL,2021-07-16,0.0000,0.0,-0.253060,0.0,1
2,AAPL,2021-07-21,0.0000,0.0,0.005334,0.0,1
3,AAPL,2021-07-26,0.5719,0.0,0.037854,0.0,1
4,AAPL,2021-07-27,0.0772,0.0,0.155385,0.0,1


##  Merge Price + Sentiment, Build Target (per ticker)

In [28]:
merged_frames = []
for t in TICKERS:
    p = price_data[t]
    s = daily_sentiment[daily_sentiment["ticker"] == t].drop(columns="ticker")
    m = pd.merge(p, s, on="date", how="left")

    # Days with no news get zero sentiment / zero count rather than being dropped —
    # dropping them would bias the dataset toward only heavily-covered news days.
    sentiment_cols = ["vader_mean", "vader_std", "finbert_mean", "finbert_std", "news_count"]
    m[sentiment_cols] = m[sentiment_cols].fillna(0)

    m["target"] = (m["Close"].shift(-1) > m["Close"]).astype(int)
    m = m.dropna(subset=["return_lag_1", "return_lag_2", "volatility_5d"])
    m = m.iloc[:-1]  # drop last row: no next-day target available
    merged_frames.append(m)

final_df = pd.concat(merged_frames, ignore_index=True)
final_df = final_df.sort_values(["ticker", "date"]).reset_index(drop=True)
print("Pooled dataset shape:", final_df.shape)
print(final_df["target"].value_counts(normalize=True))
final_df.head()

Pooled dataset shape: (6245, 13)
target
1    0.521857
0    0.478143
Name: proportion, dtype: float64


,Close,date,return_1d,return_lag_1,return_lag_2,volatility_5d,ticker,vader_mean,vader_std,finbert_mean,finbert_std,news_count,target
0,138.853409,2021-07-19,-0.026914,-0.014076,-0.004492,0.019681,AAPL,0.0,0.0,0.000000,0.0,0.0,1
1,142.460022,2021-07-20,0.025974,-0.026914,-0.014076,0.023420,AAPL,0.0,0.0,0.000000,0.0,0.0,0
2,141.728912,2021-07-21,-0.005132,0.025974,-0.026914,0.019508,AAPL,0.0,0.0,0.005334,0.0,1.0,1
3,143.093567,2021-07-22,0.009629,-0.005132,0.025974,0.020580,AAPL,0.0,0.0,0.000000,0.0,0.0,1
4,144.809143,2021-07-23,0.011989,0.009629,-0.005132,0.020084,AAPL,0.0,0.0,0.000000,0.0,0.0,1


## Train/test split

Splitting each ticker separately by date first, then combining . Otherwise a global date-based split could let one stock's later dates leak into another stock's earlier training rows.

In [29]:
train_frames, test_frames = [], []
for t in TICKERS:
    sub = final_df[final_df["ticker"] == t].sort_values("date")
    split_idx = int(len(sub) * 0.8)
    train_frames.append(sub.iloc[:split_idx])
    test_frames.append(sub.iloc[split_idx:])

train_df = pd.concat(train_frames).reset_index(drop=True)
test_df = pd.concat(test_frames).reset_index(drop=True)

print("Train size:", len(train_df), " Test size:", len(test_df))

Train size: 4995  Test size: 1250


## Baseline

Predicting the majority class. Everything below needs to beat this to actually mean anything.

In [30]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(train_df[["news_count"]], train_df["target"])  # features irrelevant, ignored by strategy
baseline_pred = baseline.predict(test_df[["news_count"]])
baseline_acc = accuracy_score(test_df["target"], baseline_pred)
print(f"Majority-class baseline accuracy: {baseline_acc:.3f}")

Majority-class baseline accuracy: 0.520


## Training the models

Running sentiment-only features and sentiment+price features separately to see if price momentum is doing more of the work than sentiment is.

In [31]:
sentiment_features = ["vader_mean", "vader_std", "finbert_mean", "finbert_std", "news_count"]
price_features = ["return_lag_1", "return_lag_2", "volatility_5d"]
combined_features = sentiment_features + price_features

def run_models(feature_set, label):
    X_train, y_train = train_df[feature_set], train_df["target"]
    X_test, y_test = test_df[feature_set], test_df["target"]

    results = {}
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
        "XGBoost": XGBClassifier(eval_metric="logloss", max_depth=3, n_estimators=200,
                                  learning_rate=0.05, random_state=42),
    }
    for name, clf in models.items():
        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)
        results[name] = {"model": clf, "preds": preds, "accuracy": acc}
        print(f"[{label}] {name}: accuracy={acc:.3f}  (baseline={baseline_acc:.3f})")
    return results

print("--- Sentiment-only features ---")
sentiment_results = run_models(sentiment_features, "Sentiment-only")

print("\n--- Sentiment + price features ---")
combined_results = run_models(combined_features, "Sentiment+Price")

--- Sentiment-only features ---
[Sentiment-only] Logistic Regression: accuracy=0.525  (baseline=0.520)
[Sentiment-only] Random Forest: accuracy=0.523  (baseline=0.520)
[Sentiment-only] XGBoost: accuracy=0.518  (baseline=0.520)

--- Sentiment + price features ---
[Sentiment+Price] Logistic Regression: accuracy=0.505  (baseline=0.520)
[Sentiment+Price] Random Forest: accuracy=0.518  (baseline=0.520)
[Sentiment+Price] XGBoost: accuracy=0.500  (baseline=0.520)


## Closer look at the best model

In [32]:
best_name = max(combined_results, key=lambda k: combined_results[k]["accuracy"])
best = combined_results[best_name]
print(f"Best model: {best_name} (accuracy={best['accuracy']:.3f}, baseline={baseline_acc:.3f})\n")
print(classification_report(test_df["target"], best["preds"]))
print("Confusion matrix:")
print(confusion_matrix(test_df["target"], best["preds"]))

Best model: Random Forest (accuracy=0.518, baseline=0.520)

              precision    recall  f1-score   support

           0       0.50      0.17      0.25       600
           1       0.52      0.84      0.65       650

    accuracy                           0.52      1250
   macro avg       0.51      0.50      0.45      1250
weighted avg       0.51      0.52      0.46      1250

Confusion matrix:
[[100 500]
 [102 548]]


## Which features actually mattered

In [33]:
rf_combined = combined_results["Random Forest"]["model"]
importances = pd.Series(rf_combined.feature_importances_, index=combined_features).sort_values(ascending=False)
importances

,0
volatility_5d,0.200166
finbert_mean,0.168744
return_lag_2,0.162623
return_lag_1,0.155734
vader_mean,0.122529
finbert_std,0.074565
vader_std,0.074517
news_count,0.041121


In [38]:
from scipy.stats import binomtest

# Is the best model's accuracy meaningfully different from the baseline rate?
n_correct = int(round(best["accuracy"] * len(test_df)))
result = binomtest(n_correct, len(test_df), p=baseline_acc, alternative="two-sided")
print(f"Best model correct: {n_correct}/{len(test_df)}")
print(f"P-value vs. baseline rate: {result.pvalue:.3f}")

Best model correct: 648/1250
P-value vs. baseline rate: 0.910


## Results

Train/test size: 4,995 train / 1,250 test (80/20 chronological split, 6,245 pooled rows across 5 tickers)

Baseline accuracy: 0.520 (majority-class)

Best accuracy (sentiment-only): 0.525 (Logistic Regression)

Best accuracy (sentiment+price): 0.518 (Random Forest) — adding price features didn't help; two of three models actually did worse than sentiment-only

Significance test result: p = 0.910 (binomial test, 648/1250 correct vs. 52.0% baseline rate) — not statistically significant


Takeaway:
Sentiment (VADER + FinBERT) showed no real edge over the majority-class baseline — best result was only 0.5 percent  above baseline, and a significance test confirmed that's just noise (p=0.91). Adding price features didn't help either. Likely due to weak news coverage (54% of days had none) and next-day direction being a genuinely hard target. A better news source or a coarser target (e.g. ±1% moves) would be the next things to try.